# Example 9.4 / 2: Write and Read One Sheet per Period

Companion notebook for **9.4 Indexed Tables and Spreadsheet Sheets** in the AMPL lab Documentation, section "Write and Read One Sheet per Period". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Write and Read One Sheet per Period

With `amplxl`, the string after the workbook name selects a sheet or named range, and it can
also depend on the index. This example writes one sheet per period to `market.xlsx` (the sheet
is created when it does not exist), clears the parameter, and reads the sheets back:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;
set PERIOD;
param market {PROD, PERIOD} >= 0;

table SheetOut {t in PERIOD} OUT "amplxl" "market.xlsx" ("P" & t) "backup=false":
    {p in PROD} -> [product], market[p, t] ~ market;
table SheetIn {t in PERIOD} IN "amplxl" "market.xlsx" ("P" & t):
    [p ~ product], market[p, t] ~ market;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROD"] = ["bands", "coils", "plate"]
ampl.set["PERIOD"] = [1, 2]
ampl.param["market"] = {("bands", 1): 6000, ("bands", 2): 5000,
                        ("coils", 1): 4000, ("coils", 2): 4500,
                        ("plate", 1): 3500, ("plate", 2): 3000}

In [5]:
# run.py
ampl.eval("load amplxl.dll;")
for t in ampl.set["PERIOD"].to_list():
    ampl.write_table(f"SheetOut[{t}]")
ampl.eval("reset data market;")
for t in ampl.set["PERIOD"].to_list():
    ampl.read_table(f"SheetIn[{t}]")
print(ampl.param["market"].to_dict())

{('bands', 1): 6000, ('bands', 2): 5000, ('coils', 1): 4000, ('coils', 2): 4500, ('plate', 1): 3500, ('plate', 2): 3000}


### Pure AMPL: data.dat and model.run

`reset data market;` has no amplpy method, so it goes through `ampl.eval`. Sheets `P1` and
`P2` each hold a `product` and a `market` column. The exact placement strings (sheet, named
range, header-only range) and options are defined by `amplxl`; see
Excel Spreadsheets (AMPL lab documentation, section 9.5.2).

Pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set PROD := bands coils plate;
set PERIOD := 1 2;
param market:  1     2 :=
  bands      6000  5000
  coils      4000  4500
  plate      3500  3000 ;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
load amplxl.dll;
for {t in PERIOD} write table SheetOut[t];
reset data market;
for {t in PERIOD} read table SheetIn[t];
display market;

market :=
bands 1   6000
bands 2   5000
coils 1   4000
coils 2   4500
plate 1   3500
plate 2   3000
;

